# Lab | LLMs through an API
## A support assistant for the bank

Over the last two days you routed the bank's customer messages with semantic search and with a fine-tuned DistilBERT. Both needed hundreds of labelled messages. The product team now asks a different question: **could a large language model do the job with no training at all, just a good prompt?** And while it is at it, could it extract the details of each message and draft the reply?

You will build that assistant with the OpenAI API on 60 real messages from the ten intents of yesterday's lab, measure its accuracy against yesterday's models, and keep track of what every step costs.

**The data** is a sample of the Banking77 dataset: `data/messages.csv` has the 60 test messages (6 per intent), and `data/examples.csv` has 20 other labelled messages (2 per intent) that you can use as examples in prompts.

**This lab practises** calling an LLM through an API. You will use, from today's **LLMs through an API** lesson: messages and roles, the system prompt, `temperature`, the `usage` field and the cost per token, zero-shot and few-shot prompts, structured outputs with Pydantic, and conversation history.

**By the end, you can:**

1. Implement an LLM-based classifier and extractor with prompts and structured outputs, and measure its accuracy and cost.
2. Explain when a prompt is enough and when a trained model is the better choice.

Each challenge ends with a **checkpoint** that tells you what you should see. The model's answers change a little from run to run, so your numbers can differ by a few points. **The whole lab costs less than 5 US cents** with the model below; the last cell shows your total.

## Setup

You need `pip install openai python-dotenv pydantic`. Create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...` (the key Ironhack gave you). The repository's `.gitignore` already lists `.env`, so git will not see it. **Never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []                               # one row per API call, so you can add up the cost at the end

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

In [ ]:
messages = pd.read_csv("../data/messages.csv")
examples = pd.read_csv("../data/examples.csv")
intents = sorted(messages["category"].unique())
print(messages.shape, examples.shape)
print(intents)

**Checkpoint:** `Key found: True`, then `(60, 2) (20, 2)` and the ten intents of yesterday's lab, from `card_arrival` to `top_up_reverted`.

## Challenge 1: a first call, and what it costs

1. Write a function `cost(input_tokens, output_tokens)` that returns the cost in US dollars with the prices in `PRICES[MODEL]`.
2. Send the first customer message (`messages["text"][0]`) to the model with a system prompt that makes it a polite bank support assistant answering in at most two sentences. Wrap the call in `track(..., "first call")`.
3. Print the reply, the number of input and output tokens, and the cost of the call.

In [ ]:
# 1. Cost of a call


# 2. First call


# 3. Reply, tokens and cost

**Checkpoint:** a short, polite reply, about 60 input and 50 output tokens, and a cost below a hundredth of a cent (about $0.00008). Read the reply: with no rules in the system prompt, the model may ask for details a bank should never ask for in a chat (in the reference run, it asked for the last four digits of the card and the billing postcode). Challenge 5 fixes that with constraints.

## Challenge 2: zero-shot intent classification

1. Write a system prompt that asks the model to classify a customer message into exactly one of the ten intents, lists the intents (you can join the list `intents` into the prompt), and asks for **the intent name only**.
2. Write `classify(text)`: one call with your system prompt and the message, `temperature=0`, tracked as `"zero-shot"`, returning the reply with spaces removed (`.strip()`).
3. Classify the 60 messages, store the predictions in a column `zero_shot`, and compute the accuracy.
4. Count the predictions that are **not** one of the ten intents.

In [ ]:
# 1. System prompt


# 2. Classifier


# 3. Predictions and accuracy


# 4. Invalid answers

**Checkpoint:** an accuracy between about 0.8 and 0.9 (0.88 in the reference solution), and no answers outside the ten intents, if your prompt asks for the name only.

In [ ]:
# 5. Show the messages that the zero-shot classifier got wrong, with the true and the predicted intent

**Question:** look at the mistakes. Which intents does the model confuse, and is it the same pattern as yesterday's models?

*Your answer here.*

## Challenge 3: few-shot classification

The intent names do not tell the model where the bank draws the line between, say, `card_arrival` and `card_delivery_estimate`. Examples can.

1. Build `few_shot_messages`: your system prompt, followed by the 20 rows of `examples` as pairs of messages, `{"role": "user", "content": text}` then `{"role": "assistant", "content": category}`.
2. Write `classify_few_shot(text)`, like `classify` but sending `few_shot_messages` plus the new message, tracked as `"few-shot"`.
3. Classify the 60 messages into a column `few_shot` and compute the accuracy.
4. From `usage_log`, compute the average number of input tokens per call for zero-shot and for few-shot, and the cost of classifying **1,000** messages with each.

In [ ]:
# 1. Few-shot messages


# 2. Classifier


# 3. Predictions and accuracy


# 4. Tokens per call and cost per 1,000 messages

**Checkpoint:** few-shot accuracy clearly higher than zero-shot (0.97 against 0.88 in the reference solution). Each few-shot call sends about 600 input tokens instead of about 100, so classifying 1,000 messages costs about 13 cents instead of about 3. The output is the same few tokens in both cases.

**Question:** compare with yesterday's lab, where zero-shot DeBERTa scored about 0.76, semantic search about 0.92 and the fine-tuned DistilBERT about 0.94 on the same ten intents. When would you use the LLM, and when the fine-tuned model? Think about accuracy, cost, speed, labels and privacy.

*Your answer here.*

## Challenge 4: structured output

The routing system needs more than the intent: it needs to know how urgent a message is and whether a person must handle it, in a format it can read without surprises.

1. Write a Pydantic class `Ticket` with four fields: `intent` (a `Literal` of the ten intents), `urgency` (`Literal["low", "medium", "high"]`), `needs_human` (`bool`, with a `Field` description saying it is true when the customer reports fraud, a lost or stolen card, or is very upset) and `summary` (`str`, at most ten words).
2. Write `extract(text)` with `client.chat.completions.parse`, the few-shot messages of Challenge 3 as context, an extra system message asking for the ticket, and `response_format=Ticket`. Track it as `"structured"` and return the parsed `Ticket`.
3. Extract a ticket for each of the 60 messages, and build a DataFrame with the message, the true intent and the four fields.
4. Compute the accuracy of `intent`, and show how many tickets need a person, and the urgency counts.

In [ ]:
# 1. Ticket


# 2. Extraction


# 3. Tickets for the 60 messages


# 4. Accuracy, people needed, urgency

**Checkpoint:** an intent accuracy close to the few-shot one (0.95 in the reference solution), every field filled with an allowed value, and about a dozen tickets (13 of 60 in the reference) that need a person, mostly fraud and lost cards.

In [ ]:
# 5. Which intents does the model send to a person most often? Show the share of needs_human per true intent.

**Question:** does `needs_human` follow the rule you gave it? Read a few tickets: would you trust the `urgency` and `needs_human` fields enough to route messages automatically? What would you check first?

*Your answer here.*

## Challenge 5: draft replies in a conversation

Finally, the assistant drafts replies for the agents. It must use the bank's real policies, and remember what the customer already said.

1. Write a system prompt with the role, the constraints (at most three sentences, never ask for the full card number, the PIN or passwords) and this context:
   - New cards arrive in 5 to 7 working days. After 10 working days, the customer can order a free replacement in the app under Card > Replace.
   - A lost or stolen card must be frozen immediately in the app under Card > Freeze; the bank then sends a new card for free.
   - A blocked PIN can be unblocked in the app under Card > PIN after confirming identity.
2. Write `reply(history, text)`: add the customer's message to `history`, call the model (tracked as `"conversation"`), add the answer as an `assistant` message, and return the answer.
3. Run this three-turn conversation and print each reply:
   - *Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.*
   - *Can I still use my old card in the meantime?*
   - *Thanks. Remind me, how many days did I say I've been waiting?*

In [ ]:
# 1. System prompt


# 2. Reply with history


# 3. Three turns

**Checkpoint:** three short replies. The first one uses the policy (more than 10 working days, so a free replacement under Card > Replace) and the customer's name. The third proves the history works: it remembers the 12 working days. No reply asks for the PIN or the card number.

**Question:** the second question (can I use my old card?) is not covered by the policies you gave. What did the assistant answer, and why is that a risk for the bank? How would you reduce it?

*Your answer here.*

## What did it cost?

Run the cell below to see the tokens and the cost of every step of the lab.

In [ ]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

**Checkpoint:** about 180 calls and a total of about 2 cents (well under 5). The structured step is the most expensive, because it sends the 20 examples with every call; the zero-shot classifier is the cheapest.

## Extra: let the assistant check the delivery status

Agents can look up where a customer's card is. Give the assistant the same ability with **function calling**.

1. Write `get_card_status(customer_id)` that looks the customer up in the dictionary `CARD_ORDERS` below and returns a dictionary with the status, or an error if the customer is unknown.
2. Describe it in a `tools` list (one required string parameter, `customer_id`, with `"strict": True`).
3. Run a conversation with the support prompt where the customer writes *My customer id is C-1043, where is my new card?*: when the model asks for the tool, run it, send the result back with the role `tool`, and print the final answer. Track every call as `"extra"`.
4. Try it with the unknown customer `C-9999`. Does the assistant handle the error well?

In [ ]:
CARD_ORDERS = {
    "C-1043": {"ordered": "2026-10-12", "status": "shipped", "carrier": "Correos", "expected": "2026-10-23"},
    "C-2210": {"ordered": "2026-10-20", "status": "being printed", "carrier": None, "expected": "2026-10-29"},
}

# Your code here

**Conclusions:** *Your answer here.*